In [1]:
import torch

from torch_openreml import MixedModelREML
from torch_openreml.covariance import DummyMatrix, ScalarMatrix

n, p = 10, 2

group = ["a", "b"] * 5
effect = torch.tensor([1.0, -1.0]).repeat(n // 2)

X = torch.randn(n, p)
y = X @ torch.tensor([0.5, -0.5]) + effect + 0.5 * torch.randn(n)

reml = MixedModelREML(
    genotype=(DummyMatrix(group), ScalarMatrix(group)),
)

names, free_params = reml.optimize(y, X, verbose=1)
names, free_params

[  1] ∥∇∥:       3.9761, ∥Δ∥: 0.9060, η: 1.00, ∥Δᶜ∥: 0.9060, accepted: True , log 𝓛:  -5.5163
[  2] ∥∇∥:       6.8866, ∥Δ∥: 0.2922, η: 1.00, ∥Δᶜ∥: 0.2922, accepted: True , log 𝓛:  -4.4322
[  3] ∥∇∥:       1.5643, ∥Δ∥: 0.1044, η: 1.00, ∥Δᶜ∥: 0.1044, accepted: True , log 𝓛:  -4.3455
[  4] ∥∇∥:       0.1552, ∥Δ∥: 0.0134, η: 1.00, ∥Δᶜ∥: 0.0134, accepted: True , log 𝓛:  -4.3445
[  5] ∥∇∥:       0.0027, ∥Δ∥: 0.0003, η: 0.25, ∥Δᶜ∥: 0.0001, accepted: True , log 𝓛:  -4.3445

[∇: score, Δ: 𝐉⁻¹∇, η: step length, Δᶜ: clip(𝛉 + ηΔ, lb, ub) - 𝛉, 𝓛: restricted likelihood]

✓ Converged at iteration 5, total time: 0.01s



(['G/sigma^2', 'R/sigma^2'], tensor([-0.0537, -0.5184]))

In [2]:
history = reml.history
history["loglik"]

[tensor(-5.5163),
 tensor(-4.4322),
 tensor(-4.3455),
 tensor(-4.3445),
 tensor(-4.3445)]

In [3]:
torch.stack(history["loglik"])

tensor([-5.5163, -4.4322, -4.3455, -4.3445, -4.3445])

In [4]:
torch.stack(history["free_params"])

tensor([[ 0.0807, -0.9024],
        [-0.0368, -0.6348],
        [-0.0494, -0.5312],
        [-0.0536, -0.5184],
        [-0.0537, -0.5184]])

In [5]:
names, theta_hat = reml.get_free_params("last")
names, theta_hat

(['G/sigma^2', 'R/sigma^2'], tensor([-0.0537, -0.5184]))

In [6]:
params = reml.build_params("last", out_format="dict")
params

{'G/sigma^2': tensor([0.8982]), 'R/sigma^2': tensor([0.3546])}

In [7]:
tree = reml.tree("last")
tree

{'/': tensor([[1.2528, 0.0000, 0.8982, 0.0000, 0.8982, 0.0000, 0.8982, 0.0000, 0.8982,
          0.0000],
         [0.0000, 1.2528, 0.0000, 0.8982, 0.0000, 0.8982, 0.0000, 0.8982, 0.0000,
          0.8982],
         [0.8982, 0.0000, 1.2528, 0.0000, 0.8982, 0.0000, 0.8982, 0.0000, 0.8982,
          0.0000],
         [0.0000, 0.8982, 0.0000, 1.2528, 0.0000, 0.8982, 0.0000, 0.8982, 0.0000,
          0.8982],
         [0.8982, 0.0000, 0.8982, 0.0000, 1.2528, 0.0000, 0.8982, 0.0000, 0.8982,
          0.0000],
         [0.0000, 0.8982, 0.0000, 0.8982, 0.0000, 1.2528, 0.0000, 0.8982, 0.0000,
          0.8982],
         [0.8982, 0.0000, 0.8982, 0.0000, 0.8982, 0.0000, 1.2528, 0.0000, 0.8982,
          0.0000],
         [0.0000, 0.8982, 0.0000, 0.8982, 0.0000, 0.8982, 0.0000, 1.2528, 0.0000,
          0.8982],
         [0.8982, 0.0000, 0.8982, 0.0000, 0.8982, 0.0000, 0.8982, 0.0000, 1.2528,
          0.0000],
         [0.0000, 0.8982, 0.0000, 0.8982, 0.0000, 0.8982, 0.0000, 0.8982, 0.0000,
    

In [8]:
beta_hat = reml.blue(y, X, theta_hat)
beta_hat

tensor([ 0.4175, -0.5698])

In [9]:
cov_beta = reml.cov_blue(X, theta_hat)
cov_beta

tensor([[ 0.0387, -0.0045],
        [-0.0045,  0.0768]])

In [10]:
se_beta = torch.sqrt(reml.var_blue(X, theta_hat))
se_beta

tensor([0.1967, 0.2772])

In [11]:
b_hat = reml.blup()
b_hat

tensor([ 0.8621, -0.9272])

In [12]:
b_cov = reml.cov_blup()
b_cov

tensor([[0.8147, 0.0084],
        [0.0084, 0.7887]])

In [13]:
b_var = reml.var_blup()
b_var

tensor([0.8147, 0.7887])

In [14]:
b_by_term = reml.blup_per_term()
b_by_term

{'genotype': tensor([ 0.8621, -0.9272])}

In [15]:
b_cov_by_term = reml.cov_blup_per_term()
b_cov_by_term

{'genotype': tensor([[0.8147, 0.0084],
         [0.0084, 0.7887]])}

In [16]:
b_var_by_term = reml.var_blup_per_term()
b_var_by_term

{'genotype': tensor([0.8147, 0.7887])}

In [17]:
y_marginal = reml.marginal_predict()
y_marginal

tensor([-0.9368,  0.5325,  0.1786, -0.6786,  0.1790, -0.2787, -0.7774, -0.4464,
         0.2060, -1.1055])

In [18]:
y_conditional = reml.predict()
y_conditional

tensor([-0.0747, -0.3947,  1.0407, -1.6058,  1.0410, -1.2059,  0.0847, -1.3736,
         1.0680, -2.0328])

In [19]:
y_new = reml.predict_new(
    X[:2],
    {"genotype": torch.tensor([[1.0, 0.0], [0.0, 1.0]])},
)
y_new

tensor([-0.0747, -0.3947])

In [20]:
z_new = DummyMatrix(group).transform(["a", "b"])

y_new = reml.predict_new(
    X[:2],
    z_new,
)
y_new

tensor([-0.0747, -0.3947])

In [21]:
e_marginal = reml.marginal_residual(y, X, theta_hat)
e_marginal

tensor([ 1.2220, -1.6707,  0.9394, -1.1729,  1.1491, -0.6263, -0.0899, -0.7994,
         1.4300, -0.7329])

In [22]:
e_conditional = reml.residual(y, X, theta_hat)
e_conditional

tensor([ 0.3600, -0.7435,  0.0773, -0.2457,  0.2870,  0.3009, -0.9519,  0.1278,
         0.5679,  0.1944])

In [23]:
c11, c12, c21, c22 = reml.c_inverse_blocks(X, theta_hat)
c11, c12, c21, c22

(tensor([[ 0.0387, -0.0045],
         [-0.0045,  0.0768]]),
 tensor([[ 0.0254, -0.0025],
         [ 0.0060, -0.0573]]),
 tensor([[ 0.0254,  0.0060],
         [-0.0025, -0.0573]]),
 tensor([[ 0.0835, -0.0084],
         [-0.0084,  0.1094]]))

In [24]:
loglik_value, loglik_method = reml.loglik(y, X, theta_hat)
loglik_value, loglik_method

(tensor(-4.3445), 'cform')

In [25]:
reml.is_converged()

True